### 0. 开始

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import utils_z
import cityjson_parser_lod1 as cjpar

In [3]:
conn = utils_z.get_conn("Test20260413", "postgres", "we6666", "localhost", "5432")

In [4]:
city_name = "berlin"
city_prefix  = "DE_BE"

### 1. 处理 lod 1 CityGML 数据并导入数据库

#### 1.1 批量转换为json

In [52]:
citygml_tools_path = r"E:\0_mylib\citygml-tools-2.4.1\citygml-tools.bat"

# 先批量转换XML到CityJSON
lod1_xml_dir = rf"E:\2_data\building_3d_opensource\{city_name}\lod1_gml"
lod1_json_dir = rf"E:\2_data\building_3d_opensource\{city_name}\lod1_json"
os.makedirs(lod1_json_dir, exist_ok=True)

In [54]:
# xml_files = [f for f in os.listdir(lod1_xml_dir) if f.endswith(".xml")]
xml_files = [f for f in os.listdir(lod1_xml_dir) if f.endswith(".gml")]
print(f"共{len(xml_files)}个文件待转换")

errors = [] 
for i, filename in enumerate(xml_files):
    input_path = os.path.join(lod1_xml_dir, filename)
    cmd = f'"{citygml_tools_path}" to-cityjson --output="{lod1_json_dir}" "{input_path}"'
    try:
        utils_z.run_cmd(cmd, False)
        if (i + 1) % 50 == 0:
            print(f"转换进度：{i + 1}/{len(xml_files)}")
    except Exception as e:
        errors.append((filename, str(e)))
        print(f"转换错误：{filename} -> {e}")

print(f"转换完成，失败{len(errors)}个")

共1个文件待转换
转换完成，失败0个


#### 1.2 检查数据，确定数据质量、srid

In [71]:
import json

test_json_path = rf"E:\2_data\building_3d_opensource\{city_name}\lod1_json\LoD1_372_5821.json"
# test_json_path = rf"E:\2_data\building_3d_opensource\{city_name}\lod2_json\8-288-536.city.json"

In [72]:
with open(test_json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# 第一段：顶层结构和对象类型统计
print("顶层keys:", data.keys())

object_counts = {}
for obj_id, obj in data["CityObjects"].items():
    obj_type = obj["type"]
    object_counts[obj_type] = object_counts.get(obj_type, 0) + 1
print("\nCityObject类型统计:")
for k, v in sorted(object_counts.items()):
    print(f"  {k}: {v}")

顶层keys: dict_keys(['type', 'version', 'CityObjects', 'transform', 'vertices', 'metadata'])

CityObject类型统计:
  Building: 397
  BuildingPart: 148


In [73]:
# 第二段：坐标系和transform
print("\nmetadata:", data.get("metadata", {}))
print("transform:", data["transform"])
print("第一个顶点原始值:", data["vertices"][0])


metadata: {'geographicalExtent': [372000.0, 5821000.0, 30.047, 373000.0, 5822000.0, 51.653]}
transform: {'scale': [0.001, 0.001, 0.001], 'translate': [372411.931, 5821119.513, 30.047]}
第一个顶点原始值: [84573, 362651, 9723]


In [74]:
# 第三段：第一个Building/BuildingPart的属性和几何

for obj_id, obj in data["CityObjects"].items():
    if obj["type"] in ("Building", "BuildingPart"):
        print(f"\n第一个对象ID: {obj_id}")
        print(f"类型: {obj['type']}")
        print(f"属性: {json.dumps(obj.get('attributes', {}), indent=2, ensure_ascii=False)}")
        for geom in obj.get("geometry", []):
            print(f"几何LOD: {geom.get('lod')}, type: {geom.get('type')}")
                  
        break



第一个对象ID: DEBE05YYY0000Pm0
类型: Building
属性: {
  "creationDate": "2019-03-01T00:00:00+08:00",
  "Gemeindeschluessel": "11000005",
  "DatenquelleDachhoehe": "1000",
  "BezugspunktDach": "2200",
  "DatenquelleLage": "1000",
  "DatenquelleBodenhoehe": "1400",
  "measuredHeight": 2.582,
  "function": "31001_1000",
  "storeysAboveGround": 1
}
几何LOD: 1, type: Solid


In [81]:
# 看LOD分布
lod_counts = {}

for obj_id, obj in data["CityObjects"].items():
    for geom in obj.get("geometry", []):
        lod = str(geom.get("lod"))
        lod_counts[lod] = lod_counts.get(lod, 0) + 1
print("LOD分布:", lod_counts)

LOD分布: {'1': 480}


In [82]:
no_geom = []
for obj_id, obj in data["CityObjects"].items():
    if obj["type"] in ("Building", "BuildingPart"):
        has_lod1 = any(str(g.get("lod")) == "1" for g in obj.get("geometry", []))
        if not has_lod1:
            no_geom.append((obj_id, obj["type"], [g.get("lod") for g in obj.get("geometry", [])]))

print(len(no_geom))

if len(no_geom) < 10:
    for item in no_geom:
        print(item)
else:
    print(f"示例ID: {no_geom[:5]}")  # 只打印前5个

65
示例ID: [('DEBE05YYY0000E2f', 'Building', []), ('DEBE05YYY0000OaX', 'Building', []), ('DEBE05YYY0000Hop', 'Building', []), ('DEBE05YYY0000FAv', 'Building', []), ('DEBE05YYY00006Ej', 'Building', [])]


#### 1.3 变量

In [5]:
block_table_name = f"blocks.{city_name}_blocks"

lod1_table_name_full = f"lod1.{city_name}_buildings_lod1"
lod1_surface_table_name_full = f"lod1.{city_name}_building_surfaces_lod1"

lod1_table_name = f"{city_name}_buildings_lod1"
lod1_surface_table_name = f"{city_name}_building_surfaces_lod1"

target_srid  = 4326
source_srid  = 25833

#### 1.4 建表

In [6]:
# LOD1建筑主表
utils_z.run_sql(f"""
    CREATE TABLE IF NOT EXISTS {lod1_table_name_full} (
        building_id     VARCHAR PRIMARY KEY,
        block_id        VARCHAR,
        citygml_id      VARCHAR,
        geom_2d         GEOMETRY(Polygon, {target_srid}),
        height          FLOAT,
        floor_count     INTEGER,
        function        VARCHAR,
        ground_z        FLOAT
    );
""", conn=conn)

utils_z.run_sql(f"""
    CREATE INDEX IF NOT EXISTS {lod1_table_name}_geom_idx
    ON {lod1_table_name_full} USING GIST (geom_2d);
""", conn=conn)

# LOD1 surface子表（无citygml_id，面由计算生成）
utils_z.run_sql(f"""
    CREATE TABLE IF NOT EXISTS {lod1_surface_table_name_full} (
        surface_id      VARCHAR PRIMARY KEY,
        building_id     VARCHAR REFERENCES {lod1_table_name_full}(building_id),
        surface_type    VARCHAR,
        geom_3d         GEOMETRY(PolygonZ, {target_srid})
    );
""", conn=conn)

utils_z.run_sql(f"""
    CREATE INDEX IF NOT EXISTS {lod1_surface_table_name}_building_idx
    ON {lod1_surface_table_name_full} (building_id);
""", conn=conn)

print(city_prefix + " LOD1表创建完成")

DE_BE LOD1表创建完成


In [19]:
# 清空building和surface表，用于需要重新处理的情况
utils_z.run_sql(f"TRUNCATE TABLE {lod1_table_name_full} CASCADE;", conn=conn)
print(f"{lod1_table_name_full}表已清空")
utils_z.run_sql(f"TRUNCATE TABLE {lod1_surface_table_name_full} CASCADE;", conn=conn)
print(f"{lod1_surface_table_name_full}表已清空")

lod1.berlin_buildings_lod1表已清空
lod1.berlin_building_surfaces_lod1表已清空


#### 1.5 批量入库

In [8]:
lod1_json_dir = rf"E:\2_data\building_3d_opensource\{city_name}\lod1_json"
# lod1_json_dir = rf"E:\2_data\building_3d_opensource\{city_name}\lod2_json"

In [9]:
import traceback

In [20]:
conn.rollback()

In [21]:
json_files = [f for f in os.listdir(lod1_json_dir) if f.endswith(".json")]
print(f"共{len(json_files)}个文件待入库")

print_interval = max(1, len(json_files) // 10) if len(json_files) > 0 else 1
print(f"打印间隔：{print_interval}")

total = 0
errors = []

# 获取当前最大ID，设置计数器初始值（空表格则为1）
cur = conn.cursor()

cur.execute(f"SELECT MAX(building_id) FROM {lod1_table_name_full}")
max_bid = cur.fetchone()[0]
building_counter = int(max_bid.split("_B_")[1]) + 1 if max_bid else 1

cur.execute(f"SELECT MAX(surface_id) FROM {lod1_surface_table_name_full}")
max_sid = cur.fetchone()[0]
surface_counter = int(max_sid.split("_S_")[1]) + 1 if max_sid else 1

cur.close()

# 遍历json文件，解析并入库，如有一个出错，整个文件跳过
for i, filename in enumerate(json_files):
    filepath = os.path.join(lod1_json_dir, filename)
    try:
        buildings = cjpar.parse_cityjson_lod1(filepath, target_lod="1")
        # buildings = cjpar.parse_cityjson_lod1_NL_AM(filepath, target_lod="1.3")
        count, building_counter, surface_counter = cjpar.insert_buildings_lod1(
            buildings, conn,
            lod1_table=lod1_table_name_full,
            surface_table=lod1_surface_table_name_full,
            city_prefix=city_prefix,
            target_srid=target_srid,
            source_srid=source_srid,
            building_counter=building_counter,
            surface_counter=surface_counter
        )
        total += count
        if (i + 1) % print_interval == 0:
            print(f"入库进度：{i+1}/{len(json_files)}，已入库建筑：{total}，已入库表面：{surface_counter-1}")
    except Exception as e:
        errors.append((filename, str(e)))
        print(f"错误：{filename} -> {e}")
        traceback.print_exc()

print(f"\n完成！共入库建筑：{total}，共入库表面：{surface_counter-1}")

print(f"失败文件数：{len(errors)}")

共1006个文件待入库
打印间隔：100
空文件，跳过：E:\2_data\building_3d_opensource\berlin\lod1_json\LoD1_370_5810.json
空文件，跳过：E:\2_data\building_3d_opensource\berlin\lod1_json\LoD1_371_5807.json
空文件，跳过：E:\2_data\building_3d_opensource\berlin\lod1_json\LoD1_371_5810.json
空文件，跳过：E:\2_data\building_3d_opensource\berlin\lod1_json\LoD1_371_5811.json
空文件，跳过：E:\2_data\building_3d_opensource\berlin\lod1_json\LoD1_371_5815.json
No ground face for building: DEBE00YY2300000d >>> E:\2_data\building_3d_opensource\berlin\lod1_json\LoD1_372_5812.json
No ground face for building: DEBE05YYR0000Apb >>> E:\2_data\building_3d_opensource\berlin\lod1_json\LoD1_372_5812.json
No ground face for building: DEBE05YYY0000STn >>> E:\2_data\building_3d_opensource\berlin\lod1_json\LoD1_372_5821.json
空文件，跳过：E:\2_data\building_3d_opensource\berlin\lod1_json\LoD1_373_5805.json
空文件，跳过：E:\2_data\building_3d_opensource\berlin\lod1_json\LoD1_373_5817.json
No ground face for building: DEBE05YYY00004eg >>> E:\2_data\building_3d_opensource\berlin\

### 2 叠合block

In [22]:
# 确认空间索引
utils_z.run_sql(f"""
    CREATE INDEX IF NOT EXISTS {lod1_table_name}_geom_idx
    ON {lod1_table_name_full} USING GIST (geom_2d);
""", conn=conn)
print("开始空间叠合...")

开始空间叠合...


In [23]:
utils_z.run_sql(f"""
    UPDATE {lod1_table_name_full} b
    SET block_id = (
        SELECT bl.block_id
        FROM {block_table_name} bl
        WHERE ST_Within(ST_Centroid(b.geom_2d), bl.geom)
        LIMIT 1
    );
""", conn=conn)
print("空间叠合完成")

空间叠合完成


In [24]:
# 检查匹配情况
result = utils_z.run_sql(f"""
    SELECT
        COUNT(*) AS total,
        COUNT(block_id) AS matched,
        COUNT(*) FILTER (WHERE block_id IS NULL) AS unmatched
    FROM {lod1_table_name_full};
""", conn=conn, fetch=True)

print(f"总建筑数：{result[0][0]}")
print(f"成功匹配block：{result[0][1]}")
print(f"未匹配block：{result[0][2]}")

总建筑数：928003
成功匹配block：844938
未匹配block：83065


In [25]:
# 检查包含LOD1建筑的街区数量
sql_counts = f"SELECT COUNT(DISTINCT block_id) FROM {lod1_table_name_full} WHERE block_id IS NOT NULL;"
(lod1_count,) = utils_z.run_sql(sql_counts, conn=conn, fetch=True)[0]

print(f"包含 LOD1 建筑的街区数量: {lod1_count}")

包含 LOD1 建筑的街区数量: 9191


### 3. 一些其他处理

##### debug: 柏林面顶点有共线情况

In [18]:
import json
import numpy as np
from shapely.geometry import Polygon

# 配置
filepath = r"E:\2_data\building_3d_opensource\berlin\lod1_json\LoD1_372_5812.json"  # 换成实际路径
target_id = "DEBE00YY2300000d"

with open(filepath, "r", encoding="utf-8") as f:
    data = json.load(f)

scale         = np.array(data["transform"]["scale"])
translate     = np.array(data["transform"]["translate"])
real_vertices = np.array(data["vertices"]) * scale + translate

obj = data["CityObjects"].get(target_id)
if obj is None:
    print("对象不存在")
else:
    geom_entry = next((g for g in obj.get("geometry", []) if str(g.get("lod")) == "1"), None)
    if geom_entry is None:
        print("没有LOD1几何")
    else:
        for shell in geom_entry["boundaries"]:
            for face in shell:
                ring   = face[0]
                coords = [tuple(real_vertices[i]) for i in ring]
                if len(coords) < 3:
                    print(f"  顶点数不足: {len(coords)}")
                    continue
                poly = Polygon(coords)
                pts  = np.array(poly.exterior.coords[:-1])
                if pts.shape[1] != 3:
                    print(f"  非3D坐标，shape={pts.shape}")
                    continue
                v1 = pts[1] - pts[0]
                v2 = pts[2] - pts[0]
                normal = np.cross(v1, v2)
                norm   = np.linalg.norm(normal)
                if norm == 0:
                    print(f"  退化面（法向量为零），顶点数={len(pts)}")
                    print("退化面顶点坐标：")
                    for i, p in enumerate(pts):
                        print(f"  [{i}] X={p[0]:.4f}  Y={p[1]:.4f}  Z={p[2]:.4f}")
                    continue
                n = normal / norm
                z_mean = np.mean(pts[:, 2])
                print(f"  normal Z: {n[2]:.4f}  z_mean: {z_mean:.3f}  顶点数: {len(pts)}")

  normal Z: 0.0000  z_mean: 49.112  顶点数: 4
  normal Z: 0.0000  z_mean: 49.112  顶点数: 4
  normal Z: 0.0000  z_mean: 49.112  顶点数: 4
  normal Z: 0.0000  z_mean: 49.112  顶点数: 4
  normal Z: 0.0000  z_mean: 49.112  顶点数: 4
  normal Z: -0.0000  z_mean: 49.112  顶点数: 4
